In [0]:
df = spark.read.parquet("/Volumes/nyc_taxi/bronze/raw_files/yellow_tripdata_2024-01.parquet")
 
df.printSchema()
print("rows:", df.count())
display(df.limit(20))

In [0]:
dropoff_earlier_count = df.filter(df.tpep_dropoff_datetime < df.tpep_pickup_datetime).count()
print("Rows where dropoff is earlier than pickup:", dropoff_earlier_count)

In [0]:
percentage = (dropoff_earlier_count / df.count()) * 100
print(f"Percentage of rows where dropoff is earlier than pickup: {percentage:.4f}%")

In [0]:
negative_fare_count = df.filter(df.fare_amount < 0).count()
zero_or_negative_distance_count = df.filter(df.trip_distance <= 0).count()
total_count = df.count()
negative_fare_pct = (negative_fare_count / total_count) * 100
zero_or_negative_distance_pct = (zero_or_negative_distance_count / total_count) * 100
print(f"Rows with negative fare_amount: {negative_fare_count} ({negative_fare_pct:.4f}%)")
print(f"Rows with zero or negative trip_distance: {zero_or_negative_distance_count} ({zero_or_negative_distance_pct:.4f}%)")

In [0]:
null_passenger_count = df.filter(df.passenger_count.isNull()).count()
zero_passenger_count = df.filter(df.passenger_count == 0).count()
missing_passenger_count = df.filter((df.passenger_count.isNull()) | (df.passenger_count == 0)).count()
total_count = df.count()

null_passenger_pct = (null_passenger_count / total_count) * 100
zero_passenger_pct = (zero_passenger_count / total_count) * 100
missing_passenger_pct = (missing_passenger_count / total_count) * 100

print(f"Rows with NULL passenger_count: {null_passenger_count} ({null_passenger_pct:.4f}%)")
print(f"Rows with 0 passenger_count: {zero_passenger_count} ({zero_passenger_pct:.4f}%)")
print(f"Rows with missing, NULL, or 0 passenger_count: {missing_passenger_count} ({missing_passenger_pct:.4f}%)")

In [0]:
from pyspark.sql import functions as F
 
print("negative fares:      ", df.filter(F.col("fare_amount") < 0).count())
print("zero/neg distance:   ", df.filter(F.col("trip_distance") <= 0).count())
print("dropoff before pickup:", df.filter(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")).count())
print("null passenger_count:", df.filter(F.col("passenger_count").isNull()).count())

In [0]:
from pyspark.sql import functions as F

# Calculate all metrics
total_count = df.count()
dropoff_earlier_count = df.filter(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")).count()
percentage = (dropoff_earlier_count / total_count) * 100
negative_fare_count = df.filter(F.col("fare_amount") < 0).count()
negative_fare_pct = (negative_fare_count / total_count) * 100
zero_or_negative_distance_count = df.filter(F.col("trip_distance") <= 0).count()
zero_or_negative_distance_pct = (zero_or_negative_distance_count / total_count) * 100
null_passenger_count = df.filter(F.col("passenger_count").isNull()).count()
null_passenger_pct = (null_passenger_count / total_count) * 100
zero_passenger_count = df.filter(F.col("passenger_count") == 0).count()
zero_passenger_pct = (zero_passenger_count / total_count) * 100
missing_passenger_count = df.filter((F.col("passenger_count").isNull()) | (F.col("passenger_count") == 0)).count()
missing_passenger_pct = (missing_passenger_count / total_count) * 100

report = """
Rows where dropoff is earlier than pickup: {}
Percentage of rows where dropoff is earlier than pickup: {:.4f}%
Rows with negative fare_amount: {} ({:.4f}%)
Rows with zero or negative trip_distance: {} ({:.4f}%)
Rows with NULL passenger_count: {} ({:.4f}%)
Rows with 0 passenger_count: {} ({:.4f}%)
Rows with missing, NULL, or 0 passenger_count: {} ({:.4f}%)
""".format(
    dropoff_earlier_count, percentage,
    negative_fare_count, negative_fare_pct,
    zero_or_negative_distance_count, zero_or_negative_distance_pct,
    null_passenger_count, null_passenger_pct,
    zero_passenger_count, zero_passenger_pct,
    missing_passenger_count, missing_passenger_pct
)

dbutils.fs.put("/Workspace/Users/rmdd.abreu@gmail.com/nyc-taxi-fabric-lakehouse/docs/data-quality-report.md", report, True)